In [1]:
!pip install trl transformers accelerate peft datasets bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 925.8/925.8 kB 34.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 MB 48.0 MB/s eta 0:00:00


In [2]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

# Load ACR guidelines + retrieval (shared with notebook 1)

In [3]:
import re, json

# Same retrieval logic as notebook 1's KD teacher-prompting step, duplicated
# here since this is a separate Kaggle session. Used to build a per-report,
# ACR-augmented system prompt for both SFT training and inference (mod #3).
ACR_GUIDELINES_PATH = "/kaggle/input/datasets/mythreyeeh/white-paper-guidelines/acr_thoracic_incidental_findings.json"

with open(ACR_GUIDELINES_PATH) as f:
    acr_guidelines = json.load(f)

def condense_finding(finding):
    feat_str = "; ".join(finding["features"])
    return f"[{finding['finding_id']}] {finding['finding_name']} \u2014 {feat_str}"

def build_condensed_index(guidelines):
    index = []
    for organ_system in guidelines:
        for finding in organ_system["findings"]:
            searchable = " ".join([finding["finding_name"], " ".join(finding["features"])]).lower()
            index.append({
                "finding_id": finding["finding_id"],
                "organ_system": organ_system["organ_system"],
                "searchable_text": searchable,
                "condensed_line": condense_finding(finding),
            })
    return index

ACR_INDEX = build_condensed_index(acr_guidelines)
full_condensed = "\n".join(f["condensed_line"] for f in ACR_INDEX)
print(f"Indexed {len(ACR_INDEX)} ACR findings ({len(full_condensed):,} chars condensed)")

STOPWORDS = {"the","a","an","of","in","on","to","and","or","with","is","are","was","were",
             "at","for","by","as","be","no","not","also","this","that","been","has","have"}

def tokenize(text):
    words = re.findall(r"[a-z]+", text.lower())
    return set(w for w in words if w not in STOPWORDS and len(w) > 2)

def retrieve_relevant_findings(report_text, index, top_k=20, min_overlap=1):
    report_tokens = tokenize(report_text)
    scored = []
    for entry in index:
        finding_tokens = tokenize(entry["searchable_text"])
        overlap = len(report_tokens & finding_tokens)
        if overlap >= min_overlap:
            scored.append((overlap, entry))
    scored.sort(key=lambda x: -x[0])
    return [entry for _, entry in scored[:top_k]]

def build_filtered_acr_context(report_text, index, top_k=10):
    relevant = retrieve_relevant_findings(report_text, index, top_k=top_k)
    if not relevant:
        return full_condensed
    return "\n".join(e["condensed_line"] for e in relevant)

TASK_INSTRUCTIONS_TEMPLATE = (
    "You are a clinical assistant specialized in thoracic radiology. Your task is to identify "
    "INCIDENTAL findings in a free-text thoracic CT report \u2014 findings unrelated to the report's "
    "primary clinical indication, per ACR Incidental Findings Committee guidelines.\n\n"
    "Use the reference guidelines below to judge whether a finding is clinically incidental "
    "(e.g. small stable nodules, benign-appearing lymph nodes, calcifications) versus a primary/"
    "expected finding tied to the report's main indication.\n\n"
    "Extract the EXACT sentence(s) from the report that describe incidental findings \u2014 do not "
    "paraphrase or summarize. If no incidental findings are present, return an empty list.\n\n"
    "REFERENCE GUIDELINES:\n{acr_context}"
)

def build_system_prompt(report_text, top_k=10):
    filtered_context = build_filtered_acr_context(report_text, ACR_INDEX, top_k=top_k)
    return TASK_INSTRUCTIONS_TEMPLATE.format(acr_context=filtered_context)

Indexed 41 ACR findings (12,750 chars condensed)


# Build SFT training data (report -> ACR-augmented prompt -> gold JSON)

In [4]:
import json
import os
import gc

# Same dataset your original QLoRA run used, path corrected to match the
# actual Kaggle input mount (mythreyeeh/train-dataset-thoractic).
unannotated_source = "/kaggle/input/datasets/mythreyeeh/train-dataset-thoractic/Train_dataset_thoractic/dataset_thoracic_unannotated.json"
annotated_source = "/kaggle/input/datasets/mythreyeeh/train-dataset-thoractic/Train_dataset_thoractic/dataset_thoracic_annotated.json"
output_file = "train_chat.jsonl"

# 1. Load the files
with open(unannotated_source, "r", encoding="utf-8") as f:
    unannotated_reports = json.load(f)["reports"]

with open(annotated_source, "r", encoding="utf-8") as f:
    annotated_reports = json.load(f)["reports"]

# 2. Fast lookup map of annotations using report_id as key
annotation_lookup = {r["report_id"]: r["annotation"] for r in annotated_reports}

# 3. Build BOTH the SFT chat-format records AND a parallel "structured" record
#    (report_id, free_text, gold) reused later for eval, so train/val split
#    stays identical between SFT data and eval data.
formatted_records = []
structured_records = []
matched_count = 0

for report in unannotated_reports:
    rid = report["report_id"]
    free_text = report["free_text"]

    if rid in annotation_lookup:
        gold_annotation = annotation_lookup[rid]
        matched_count += 1

        # mod #3: ACR context is now baked into a per-report system prompt,
        # instead of the flat generic SYSTEM_PROMPT the original run used.
        system_prompt = build_system_prompt(free_text)

        chatml_structure = {
            "report_id": rid,
            "contains_IF": gold_annotation["contains_IF"],
            "messages": [
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": f"Report:\n{free_text}"},
                {
                    "role": "assistant",
                    "content": json.dumps({
                        "contains_IF": gold_annotation["contains_IF"],
                        "incidental_sentences": gold_annotation["incidental_sentences"],
                    }),
                },
            ],
        }
        formatted_records.append(chatml_structure)

        structured_records.append({
            "report_id": rid,
            "free_text": free_text,
            "gold": {
                "contains_IF": gold_annotation["contains_IF"],
                "incidental_sentences": gold_annotation["incidental_sentences"],
            },
        })

# 4. Write out JSONL
with open(output_file, "w", encoding="utf-8") as f:
    for record in formatted_records:
        f.write(json.dumps(record) + "\n")

print("Mapping complete!")
print(f"Successfully paired {matched_count} annotated files out of {len(unannotated_reports)} total reports.")
print(f"Saved training-ready format to: {output_file}")

Mapping complete!
Successfully paired 1000 annotated files out of 1000 total reports.
Saved training-ready format to: train_chat.jsonl


# Load merged-KD base model + tokenizer, split data

In [5]:
import torch
import torch.nn.functional as F
import wandb
import numpy as np
from datasets import Dataset
from transformers import (
    AutoModelForCausalLM, AutoTokenizer,
    TrainingArguments, TrainerCallback, TrainerState, TrainerControl,
    BitsAndBytesConfig,
)
from trl import SFTTrainer, SFTConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from sklearn.model_selection import train_test_split

EVAL_BATCH_SIZE = 8
EVAL_STEPS = 50
FOCAL_GAMMA = 2.0

wandb_key = os.environ.get("WANDB_API_KEY")
if wandb_key:
    wandb.login(key=wandb_key)
else:
    os.environ["WANDB_MODE"] = "disabled"
    print("WANDB_API_KEY not set -- running with wandb disabled.")

model_id = "/kaggle/input/datasets/mythreyeeh/kd-best-thoracic"

tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

# --- Split ONCE, reuse the same split for SFT text and eval ---
train_structured, val_structured = train_test_split(
    structured_records, test_size=100, random_state=42
)
print(f"Train: {len(train_structured)}, Val: {len(val_structured)}")

train_ids = {r["report_id"] for r in train_structured}
val_ids = {r["report_id"] for r in val_structured}

train_records = [r for r in formatted_records if r["report_id"] in train_ids]
val_records_chat = [r for r in formatted_records if r["report_id"] in val_ids]

# --- Class weights, computed from the actual train split ---
n_pos = sum(1 for r in train_records if r["contains_IF"])
n_neg = len(train_records) - n_pos
n_total = len(train_records)
class_weight = {
    True:  n_total / (2 * n_pos),
    False: n_total / (2 * n_neg),
}
print(f"n_pos={n_pos}, n_neg={n_neg}, class_weight={class_weight}")

# --- Tokenize with left-truncation (never clips the gold completion) ---
MAX_LEN = 3072

def tokenize_with_labels(example):
    system_msg, user_msg, assistant_msg = example["messages"]
    prompt_text = tokenizer.apply_chat_template(
        [system_msg, user_msg], tokenize=False, add_generation_prompt=True
    )
    full_text = tokenizer.apply_chat_template(
        example["messages"], tokenize=False, add_generation_prompt=False
    )
    prompt_ids = tokenizer(prompt_text, add_special_tokens=False)["input_ids"]
    full_ids = tokenizer(full_text, add_special_tokens=False)["input_ids"]

    if len(full_ids) > MAX_LEN:
        overflow = len(full_ids) - MAX_LEN
        full_ids = full_ids[overflow:]
        prompt_len = max(len(prompt_ids) - overflow, 0)
    else:
        prompt_len = len(prompt_ids)

    labels = [-100] * prompt_len + full_ids[prompt_len:]
    return {
        "input_ids": full_ids,
        "attention_mask": [1] * len(full_ids),
        "labels": labels,
        "class_weight": class_weight[example["contains_IF"]],
    }

train_dataset = Dataset.from_list(train_records).map(
    tokenize_with_labels,
    remove_columns=["messages", "report_id", "contains_IF"],
)

token_lengths = [len(ex) for ex in train_dataset["input_ids"]]
print(f"Token lengths -- min: {min(token_lengths)}, max: {max(token_lengths)}, "
      f"mean: {np.mean(token_lengths):.0f}, p95: {np.percentile(token_lengths, 95):.0f}")

WANDB_API_KEY not set -- running with wandb disabled.
Train: 900, Val: 100
n_pos=667, n_neg=233, class_weight={True: 0.6746626686656672, False: 1.9313304721030042}


Map:   0%|          | 0/900 [00:00<?, ? examples/s]

Token lengths -- min: 903, max: 2027, mean: 1316, p95: 1611


# Eval helpers (fuzzy matching) + early-stopping callback

In [6]:
import re
from difflib import SequenceMatcher


def parse_output(raw_text):
    try:
        return json.loads(raw_text)
    except json.JSONDecodeError:
        pass
    match = re.search(r"\{.*\}", raw_text, re.DOTALL)
    if match:
        try:
            return json.loads(match.group())
        except json.JSONDecodeError:
            pass
    return None


def build_messages(record, few_shot_pool=None, n_shot=0):
    """mod #3: ACR-augmented system prompt is built per-record instead of a
    single flat SYSTEM_PROMPT."""
    messages = [{"role": "system", "content": build_system_prompt(record["free_text"])}]

    if few_shot_pool and n_shot > 0:
        for ex in few_shot_pool[:n_shot]:
            messages.append({"role": "user", "content": f"Report:\n{ex['free_text']}"})
            messages.append({
                "role": "assistant",
                "content": json.dumps({
                    "contains_IF": ex["gold"]["contains_IF"],
                    "incidental_sentences": ex["gold"]["incidental_sentences"],
                }),
            })

    messages.append({"role": "user", "content": f"Report:\n{record['free_text']}"})
    return messages


# --- mod #1: fuzzy matching eval, replaces exact-set-overlap scoring ---
def similarity(a, b):
    return SequenceMatcher(None, a, b).ratio()

def fuzzy_match_sets(gold_set, pred_set, threshold=0.85):
    """Greedy one-to-one fuzzy matching between gold and predicted sentences."""
    gold_list = list(gold_set)
    pred_list = list(pred_set)
    matched_gold, matched_pred = set(), set()
    pairs = []
    for gi, g in enumerate(gold_list):
        for pi, p in enumerate(pred_list):
            sim = similarity(g, p)
            if sim >= threshold:
                pairs.append((sim, gi, pi))
    pairs.sort(key=lambda x: -x[0])
    for sim, gi, pi in pairs:
        if gi in matched_gold or pi in matched_pred:
            continue
        matched_gold.add(gi)
        matched_pred.add(pi)
    tp = len(matched_gold)
    fp = len(pred_list) - len(matched_pred)
    fn = len(gold_list) - len(matched_gold)
    return tp, fp, fn


def run_evaluation_fuzzy(results, label="", threshold=0.85):
    parse_failures = sum(r["parse_failed"] for r in results)
    valid = [r for r in results if not r["parse_failed"]]
    total_tp = total_fp = total_fn = 0
    neg_scores, pos_scores = [], []
    for r in valid:
        gold_set = set(s.strip().lower() for s in r["gold_sentences"])
        pred_set = set(s.strip().lower() for s in (r["pred_sentences"] or []))
        if len(gold_set) == 0:
            neg_scores.append(1.0 if len(pred_set) == 0 else 0.0)
        else:
            tp, fp, fn = fuzzy_match_sets(gold_set, pred_set, threshold=threshold)
            total_tp += tp; total_fp += fp; total_fn += fn
            precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
            recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
            f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
            pos_scores.append(f1)
    avg_neg = sum(neg_scores) / len(neg_scores) if neg_scores else 0.0
    avg_pos = sum(pos_scores) / len(pos_scores) if pos_scores else 0.0
    n_neg, n_pos = len(neg_scores), len(pos_scores)
    if n_neg > 0 and n_pos > 0:
        macro_f1 = (avg_neg + avg_pos) / 2
    elif n_neg > 0:
        macro_f1 = avg_neg
    elif n_pos > 0:
        macro_f1 = avg_pos
    else:
        macro_f1 = 0.0
    weighted_f1 = (n_neg * avg_neg + n_pos * avg_pos) / (n_neg + n_pos) if (n_neg + n_pos) > 0 else 0.0
    micro_precision = total_tp / (total_tp + total_fp) if (total_tp + total_fp) > 0 else 0.0
    micro_recall = total_tp / (total_tp + total_fn) if (total_tp + total_fn) > 0 else 0.0
    micro_f1 = (2 * micro_precision * micro_recall / (micro_precision + micro_recall)
                if (micro_precision + micro_recall) > 0 else 0.0)
    print(f"\n{'='*50}\n=== {label} (fuzzy threshold={threshold}) ===\n{'='*50}")
    print(f"Parse failures:      {parse_failures}/{len(results)}")
    print(f"Negative-report acc: {avg_neg:.4f}  (n={n_neg})")
    print(f"Positive-report F1:  {avg_pos:.4f}  (n={n_pos})")
    print(f"Sentence Macro F1:   {macro_f1:.4f}")
    print(f"Sentence Weighted:   {weighted_f1:.4f}")
    print(f"Sentence Micro F1:   {micro_f1:.4f}")
    return {"macro_f1": macro_f1, "weighted_f1": weighted_f1, "micro_f1": micro_f1,
            "parse_failures": parse_failures, "n_neg": n_neg, "n_pos": n_pos,
            "avg_neg": avg_neg, "avg_pos": avg_pos}


def generate_predictions(model, tokenizer, records, few_shot_pool=None, n_shot=0,
                          n=None, batch_size=EVAL_BATCH_SIZE):
    import transformers
    transformers.logging.set_verbosity_error()
    model.eval()

    subset = records[:n] if n else records
    results = []

    for start in range(0, len(subset), batch_size):
        batch_records = subset[start:start + batch_size]
        texts = [
            tokenizer.apply_chat_template(
                build_messages(r, few_shot_pool, n_shot),
                tokenize=False, add_generation_prompt=True
            )
            for r in batch_records
        ]
        inputs = tokenizer(texts, return_tensors="pt", padding=True, truncation=True).to(model.device)

        with torch.no_grad():
            output_ids = model.generate(
                inputs["input_ids"],
                attention_mask=inputs["attention_mask"],
                max_new_tokens=256,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
            )

        prompt_len = inputs["input_ids"].shape[-1]
        for i, record in enumerate(batch_records):
            generated = output_ids[i][prompt_len:]
            raw = tokenizer.decode(generated, skip_special_tokens=True).strip()
            parsed = parse_output(raw)

            results.append({
                "report_id": record["report_id"],
                "gold_sentences": record["gold"]["incidental_sentences"],
                "pred_sentences": parsed.get("incidental_sentences", []) if parsed else None,
                "parse_failed": parsed is None,
            })

    return results


class MacroF1EarlyStoppingCallback(TrainerCallback):
    def __init__(self, eval_records, tokenizer, run_name, eval_steps=EVAL_STEPS, patience=3):
        self.eval_records = eval_records
        self.tokenizer = tokenizer
        self.run_name = run_name
        self.eval_steps = eval_steps
        self.patience = patience
        self.best_f1 = -1.0
        self.no_improve = 0
        self.best_step = 0
        self.best_metrics = None
        self.best_ckpt_dir = f"./qlora_best_{run_name}"

    def on_step_end(self, args, state, control, **kwargs):
        if state.global_step % self.eval_steps != 0 or state.global_step == 0:
            return control
    
        model = kwargs["model"]
        results = generate_predictions(model, self.tokenizer, self.eval_records, n=100)
        metrics = run_evaluation_fuzzy(results, label=f"Step {state.global_step}")
    
        wandb.log({
            "val_macro_f1": metrics["macro_f1"],
            "val_micro_f1": metrics["micro_f1"],
            "val_weighted_f1": metrics["weighted_f1"],
            "val_avg_neg": metrics["avg_neg"],
            "val_avg_pos": metrics["avg_pos"],
            "val_parse_failures": metrics["parse_failures"],
            "step": state.global_step,
        })
    
        if metrics["macro_f1"] > self.best_f1:
            self.best_f1 = metrics["macro_f1"]
            self.best_step = state.global_step
            self.best_metrics = metrics
            self.no_improve = 0
            model.save_pretrained(self.best_ckpt_dir)
            self.tokenizer.save_pretrained(self.best_ckpt_dir)
            print(f"  New best saved -> {self.best_ckpt_dir}")
        else:
            self.no_improve += 1
            print(f"  No improvement ({self.no_improve}/{self.patience})")
    
        if self.no_improve >= self.patience:
            print(f"\nEarly stopping at step {state.global_step}. Best step {self.best_step}, macro F1={self.best_f1:.4f}")
            control.should_training_stop = True
    
        return control

# QLoRA training with focal loss, on top of the merged KD checkpoint

In [7]:
class WeightedDataCollator:
    def __init__(self, tokenizer):
        self.pad_token_id = tokenizer.pad_token_id

    def __call__(self, features):
        max_len = max(len(f["input_ids"]) for f in features)
        input_ids, attention_mask, labels, weights = [], [], [], []
        for f in features:
            pad_n = max_len - len(f["input_ids"])
            input_ids.append(f["input_ids"] + [self.pad_token_id] * pad_n)
            attention_mask.append(f["attention_mask"] + [0] * pad_n)
            labels.append(f["labels"] + [-100] * pad_n)
            weights.append(f["class_weight"])
        return {
            "input_ids": torch.tensor(input_ids, dtype=torch.long),
            "attention_mask": torch.tensor(attention_mask, dtype=torch.long),
            "labels": torch.tensor(labels, dtype=torch.long),
            "class_weight": torch.tensor(weights, dtype=torch.float32),
        }

In [8]:
# Hyperparameter grid -- unchanged from your original run
lora_configs = [
    {"r": 32, "lora_alpha": 64},
]

sweep_results = []

# 4-bit NF4 quantization config.
# Changed from FP16 compute to BF16 to test/fix the NaN logits
# observed with the FP16 quantized forward pass.
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)


class WeightedCETrainer(SFTTrainer):
    """Plain CE with a per-example class weight."""
    def compute_loss(
        self,
        model,
        inputs,
        return_outputs=False,
        num_items_in_batch=None,
    ):
        labels = inputs.pop("labels")
        weights = inputs.pop("class_weight")

        outputs = model(**inputs)
        logits = outputs.logits

        shift_logits = logits[..., :-1, :].contiguous()
        shift_labels = labels[..., 1:].contiguous()

        log_probs = F.log_softmax(shift_logits.float(), dim=-1)

        B, L, V = log_probs.shape

        ce = F.nll_loss(
            log_probs.view(-1, V),
            shift_labels.view(-1),
            ignore_index=-100,
            reduction="none",
        ).view(B, L)

        valid_mask = (shift_labels != -100).float()

        per_example_weight = (
            weights
            .to(ce.device, ce.dtype)
            .unsqueeze(1)
        )

        loss = (
            (ce * valid_mask * per_example_weight).sum()
            /
            (valid_mask * per_example_weight)
            .sum()
            .clamp(min=1e-8)
        )

        return (loss, outputs) if return_outputs else loss


class FocalLossSFTTrainer(SFTTrainer):
    """Focal loss with class-weighted alpha.
    Kept as a fallback; not used in this run.
    """
    def compute_loss(
        self,
        model,
        inputs,
        return_outputs=False,
        num_items_in_batch=None,
    ):
        labels = inputs.pop("labels")
        weights = inputs.pop("class_weight")

        outputs = model(**inputs)
        logits = outputs.logits

        shift_logits = logits[..., :-1, :].contiguous()
        shift_labels = labels[..., 1:].contiguous()

        log_probs = F.log_softmax(shift_logits.float(), dim=-1)

        B, L, V = log_probs.shape

        ce = F.nll_loss(
            log_probs.view(-1, V),
            shift_labels.view(-1),
            ignore_index=-100,
            reduction="none",
        ).view(B, L)

        valid_mask = (shift_labels != -100).float()

        with torch.no_grad():
            pt = (
                log_probs.view(-1, V)
                .gather(
                    1,
                    shift_labels.view(-1)
                    .clamp(min=0)
                    .unsqueeze(1),
                )
                .squeeze(1)
                .exp()
                .view(B, L)
            )

        focal_term = (1 - pt).pow(FOCAL_GAMMA)

        per_example_weight = (
            weights
            .to(ce.device, ce.dtype)
            .unsqueeze(1)
        )

        loss = (
            (focal_term * ce * valid_mask * per_example_weight).sum()
            /
            (valid_mask * per_example_weight)
            .sum()
            .clamp(min=1e-8)
        )

        return (loss, outputs) if return_outputs else loss


for config in lora_configs:
    run_name = f"r{config['r']}_alpha{config['lora_alpha']}"

    print(f"\n{'=' * 50}")
    print(f"Starting run: {run_name}")
    print(f"{'=' * 50}")

    wandb.init(
        project="incidental-findings-finetuning",
        name=f"kd-then-qlora-{run_name}",
        config=config,
        reinit=True,
    )

    # Load the merged KD checkpoint.
    # This is QLoRA on top of KD, not QLoRA from the original Qwen model.
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        quantization_config=bnb_config,
        torch_dtype=torch.bfloat16,
        device_map={"": 0},
    )

    model = prepare_model_for_kbit_training(model)

    print("Model config torch_dtype:", model.config.torch_dtype)

    for name, param in model.named_parameters():
        if param.requires_grad:
            print(
                "First trainable parameter:",
                name,
                param.dtype,
            )
            break

    peft_config = LoraConfig(
        r=config["r"],
        lora_alpha=config["lora_alpha"],
        target_modules=[
            "q_proj",
            "k_proj",
            "v_proj",
            "o_proj",
        ],
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM",
    )

    training_args = SFTConfig(
        output_dir=f"./qwen_lora_{run_name}",
        per_device_train_batch_size=2,
        gradient_accumulation_steps=8,
        max_length=3072,
        learning_rate=2e-4,
        logging_steps=10,
        num_train_epochs=100,
        save_strategy="steps",
        save_steps=50,
        eval_strategy="no",
        lr_scheduler_type="cosine",
        warmup_steps=10,
        fp16=False,
        bf16=True,
        report_to="wandb",
        remove_unused_columns=False,
        dataset_kwargs={
            "skip_prepare_dataset": True
        },
        gradient_checkpointing=True,
    )

    callback = MacroF1EarlyStoppingCallback(
        eval_records=val_structured,
        tokenizer=tokenizer,
        run_name=run_name,
        eval_steps=50,
        patience=3,
    )

    trainer = WeightedCETrainer(
        model=model,
        train_dataset=train_dataset,
        peft_config=peft_config,
        processing_class=tokenizer,
        args=training_args,
        callbacks=[callback],
        data_collator=WeightedDataCollator(tokenizer),
    )

    print("Training examples:", len(train_dataset))

    # Keep LoRA parameters in BF16 for this BF16 test.
    # Do NOT convert them to FP16 here.

    dtype_counts = {}

    for name, param in trainer.model.named_parameters():
        if param.requires_grad:
            dtype_counts.setdefault(
                param.dtype,
                []
            ).append(name)

    print("\nTrainable parameter dtypes:")

    for dtype, names in dtype_counts.items():
        print(
            f"{dtype}: "
            f"{len(names)} trainable params, "
            f"e.g. {names[0]}"
        )

    # ---------------------------------------------------------
    # Verify the custom batch/collator
    # ---------------------------------------------------------

    batch = next(iter(trainer.get_train_dataloader()))

    print("\nBatch keys:", batch.keys())

    assert "class_weight" in batch, \
        "class_weight was dropped again!"

    print(
        "class_weight sample:",
        batch["class_weight"][:5]
    )

    # ---------------------------------------------------------
    # IMPORTANT:
    # Test the forward pass BEFORE training.
    # The previous FP16 setup produced NaN logits.
    # ---------------------------------------------------------

    print("\nChecking forward pass before training...")

    with torch.no_grad():
        outputs = trainer.model(
            input_ids=batch["input_ids"].to(model.device),
            attention_mask=batch["attention_mask"].to(model.device),
        )

    logits = outputs.logits

    print("Logits dtype:", logits.dtype)
    print(
        "Logits NaN:",
        torch.isnan(logits).sum().item()
    )
    print(
        "Logits Inf:",
        torch.isinf(logits).sum().item()
    )

    if torch.isnan(logits).any():
        raise RuntimeError(
            "NaN logits detected before training. "
            "BF16 did not resolve the numerical instability."
        )

    if torch.isinf(logits).any():
        raise RuntimeError(
            "Inf logits detected before training."
        )

    print("Forward pass is numerically stable.")

    # Free the temporary forward-pass outputs before training.
    del outputs, logits
    gc.collect()
    torch.cuda.empty_cache()

    # ---------------------------------------------------------
    # Start training
    # ---------------------------------------------------------

    trainer.train()

    # ---------------------------------------------------------
    # Store best result
    # ---------------------------------------------------------

    sweep_results.append({
        "run": run_name,
        "r": config["r"],
        "lora_alpha": config["lora_alpha"],
        "best_f1": callback.best_f1,
        "best_step": callback.best_step,
    })

    wandb.finish()

    # ---------------------------------------------------------
    # Free GPU memory before next sweep iteration
    # ---------------------------------------------------------

    del trainer, model
    gc.collect()
    torch.cuda.empty_cache()


# -------------------------------------------------------------
# Summary
# -------------------------------------------------------------

print("\n" + "=" * 50)
print("Sweep Summary:")
print("=" * 50)

for r in sorted(
    sweep_results,
    key=lambda x: x["best_f1"],
    reverse=True,
):
    print(
        f"  {r['run']:20s} | "
        f"Best F1: {r['best_f1']:.4f} | "
        f"Step: {r['best_step']}"
    )

`torch_dtype` is deprecated! Use `dtype` instead!



Starting run: r32_alpha64


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Model config torch_dtype: torch.bfloat16
Training examples: 900

Trainable parameter dtypes:
torch.bfloat16: 192 trainable params, e.g. base_model.model.model.layers.0.self_attn.q_proj.lora_A.default.weight

Batch keys: dict_keys(['input_ids', 'attention_mask', 'labels', 'class_weight'])
class_weight sample: tensor([0.6747, 0.6747], device='cuda:0')

Checking forward pass before training...
Logits dtype: torch.float32
Logits NaN: 0
Logits Inf: 0
Forward pass is numerically stable.


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151645}.


Step,Training Loss
10,0.734153
20,0.633305
30,0.595240
40,0.489857
50,0.460529
60,0.360654
70,0.277934
80,0.315447
90,0.260634
100,0.292610



=== Step 50 (fuzzy threshold=0.85) ===
Parse failures:      0/100
Negative-report acc: 0.7500  (n=24)
Positive-report F1:  0.7088  (n=76)
Sentence Macro F1:   0.7294
Sentence Weighted:   0.7187
Sentence Micro F1:   0.6950
  New best saved -> ./qlora_best_r32_alpha64

=== Step 100 (fuzzy threshold=0.85) ===
Parse failures:      0/100
Negative-report acc: 0.8750  (n=24)
Positive-report F1:  0.7023  (n=76)
Sentence Macro F1:   0.7886
Sentence Weighted:   0.7437
Sentence Micro F1:   0.7103
  New best saved -> ./qlora_best_r32_alpha64

=== Step 150 (fuzzy threshold=0.85) ===
Parse failures:      1/100
Negative-report acc: 0.7500  (n=24)
Positive-report F1:  0.7583  (n=75)
Sentence Macro F1:   0.7542
Sentence Weighted:   0.7563
Sentence Micro F1:   0.7554
  No improvement (1/3)

=== Step 200 (fuzzy threshold=0.85) ===
Parse failures:      0/100
Negative-report acc: 0.7917  (n=24)
Positive-report F1:  0.7630  (n=76)
Sentence Macro F1:   0.7773
Sentence Weighted:   0.7699
Sentence Micro F1:  

# Standalone evaluation (run after training)

In [9]:
# Standalone evaluation script.
# Run this AFTER training -- it does not train anything, it just loads a
# saved LoRA checkpoint (on top of the merged KD base) and scores it against
# the held-out test set with fuzzy matching (mod #1).

import json
import re
import gc
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

# --- Config: point this at whichever checkpoint you want to evaluate ---
checkpoint_dir = "./qlora_best_r32_alpha64"   # <- change to the run you want to evaluate

test_unannotated_source = "/kaggle/input/datasets/mythreyeeh/test-dataset-thoractic/Test_dataset_thoractic/test_dataset_thoracic_unannotated.json"
test_annotated_source = "/kaggle/input/datasets/mythreyeeh/test-dataset-thoractic/Test_dataset_thoractic/test_dataset_thoracic_annotated.json"

EVAL_BATCH_SIZE = 8

In [10]:
# --- Load test set ---
with open(test_unannotated_source, "r", encoding="utf-8") as f:
    test_unannotated_reports = json.load(f)["reports"]

with open(test_annotated_source, "r", encoding="utf-8") as f:
    test_annotated_reports = json.load(f)["reports"]

test_annotation_lookup = {r["report_id"]: r["annotation"] for r in test_annotated_reports}

test_structured_records = []
for report in test_unannotated_reports:
    rid = report["report_id"]
    if rid in test_annotation_lookup:
        gold_annotation = test_annotation_lookup[rid]
        test_structured_records.append({
            "report_id": rid,
            "free_text": report["free_text"],
            "gold": {
                "contains_IF": gold_annotation["contains_IF"],
                "incidental_sentences": gold_annotation["incidental_sentences"],
            },
        })

print(f"Loaded {len(test_structured_records)} test records "
      f"(of {len(test_unannotated_reports)} total, matched to annotations).")

Loaded 100 test records (of 100 total, matched to annotations).


In [11]:
# --- Load tokenizer + quantized (merged-KD) base model + QLoRA adapter ---
tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"   # required for correct batched generation

# Must match the quantization the adapter was trained under (4-bit nf4).
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

base_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map={"": 0},
)
model = PeftModel.from_pretrained(base_model, checkpoint_dir)
model.eval()

# --- run_eval / parse_output / build_messages / fuzzy_match_sets / similarity /
# build_system_prompt are all reused from earlier cells in this notebook. ---

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen2ForCausalLM(
      (model): Qwen2Model(
        (embed_tokens): Embedding(151936, 896)
        (layers): ModuleList(
          (0-23): 24 x Qwen2DecoderLayer(
            (self_attn): Qwen2Attention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=896, out_features=896, bias=True)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=896, out_features=32, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=32, out_features=896, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora.Line

In [12]:
# --- Run it ---
results = generate_predictions(model, tokenizer, test_structured_records, n=None)
metrics = run_evaluation_fuzzy(results, label="Test Set")

del model, base_model
gc.collect()
torch.cuda.empty_cache()


=== Test Set (fuzzy threshold=0.85) ===
Parse failures:      2/100
Negative-report acc: 0.7600  (n=25)
Positive-report F1:  0.6757  (n=73)
Sentence Macro F1:   0.7179
Sentence Weighted:   0.6972
Sentence Micro F1:   0.6667
